# Global Automotive Investment Database — Block 6

## Korea: OpenDART point-in-time fundamentals

Loads the persisted Security Master and shared canonical schema, resolves Korean
stock codes to DART corporation codes, retrieves filing histories and full
financial-statement accounts, maps them to the common schema, and persists a
Block 6 manifest.

Create a Colab secret named `DART_API_KEY` before running.

In [ ]:
# 1. IMPORTS

!pip -q install pandas numpy requests tqdm pyarrow lxml

from __future__ import annotations
import hashlib, json, os, re, time, zipfile
from datetime import datetime, timezone
from io import BytesIO
from pathlib import Path
from typing import Iterable, Optional
from xml.etree import ElementTree as ET

import numpy as np
import pandas as pd
import requests
from tqdm.auto import tqdm

pd.set_option("display.max_columns", 250)
pd.set_option("display.width", 260)

In [ ]:
# 2. SETTINGS, SECRETS AND UPSTREAM INPUTS

USE_GOOGLE_DRIVE = True

if USE_GOOGLE_DRIVE:
    from google.colab import drive, userdata
    drive.mount("/content/drive")
    PROJECT_ROOT = Path("/content/drive/MyDrive/Colab Notebooks/00 A1 Auto Factor Strategy")
    try:
        DART_API_KEY = userdata.get("DART_API_KEY").strip()
    except Exception as exc:
        raise ValueError(
            "Create a Colab secret named 'DART_API_KEY' and enable notebook access."
        ) from exc
else:
    PROJECT_ROOT = Path("/content/global_automotive_investment_database")
    DART_API_KEY = os.environ.get("DART_API_KEY", "").strip()

if not DART_API_KEY:
    raise ValueError("DART_API_KEY is empty.")

DATA_ROOT = PROJECT_ROOT / "data"
BLOCK_2_MANIFEST_PATH = DATA_ROOT / "interim" / "block_2" / "block_2_manifest.json"
BLOCK_4_MANIFEST_PATH = DATA_ROOT / "interim" / "block_4" / "block_4_manifest.json"
BLOCK_6_OUTPUT_DIR = DATA_ROOT / "interim" / "block_6"
BLOCK_6_MANIFEST_PATH = BLOCK_6_OUTPUT_DIR / "block_6_manifest.json"

DART_RAW_DIR = DATA_ROOT / "raw" / "dart"
DART_CORP_CACHE_DIR = DART_RAW_DIR / "corp_codes"
DART_FILING_CACHE_DIR = DART_RAW_DIR / "filings"
DART_FINANCIAL_CACHE_DIR = DART_RAW_DIR / "financials"

for d in [BLOCK_6_OUTPUT_DIR, DART_CORP_CACHE_DIR, DART_FILING_CACHE_DIR, DART_FINANCIAL_CACHE_DIR]:
    d.mkdir(parents=True, exist_ok=True)

DART_API_BASE = "https://opendart.fss.or.kr/api"
DISCOVERY_START_DATE = "2019-10-01"
DISCOVERY_END_DATE = pd.Timestamp.today().strftime("%Y-%m-%d")
REPORT_CODES = {"11011":"ANNUAL","11012":"SEMIANNUAL","11013":"Q1","11014":"Q3"}
FS_DIVISIONS = {"CFS":"CONSOLIDATED","OFS":"SEPARATE"}

REQUEST_INTERVAL_SECONDS = 0.20
REQUEST_TIMEOUT_SECONDS = 120
MAX_RETRIES = 5
MAX_CORPORATIONS = None
MAX_FINANCIAL_REQUESTS = None
PERSIST_BLOCK_6_OUTPUTS = True
OVERWRITE_PERSISTED_OUTPUTS = True

def load_manifest_tables(manifest_path, names):
    with manifest_path.open("r", encoding="utf-8") as f:
        manifest = json.load(f)
    records = {x["table_name"]: x for x in manifest["tables"]}
    missing = set(names) - set(records)
    if missing:
        raise RuntimeError(f"{manifest_path.name} missing {sorted(missing)}")
    loaded = {}
    for name in names:
        path = Path(records[name]["path"])
        if not path.exists():
            raise FileNotFoundError(path)
        loaded[name] = pd.read_parquet(path)
    return loaded, manifest

b2, block_2_manifest = load_manifest_tables(
    BLOCK_2_MANIFEST_PATH,
    {"security_master_df","issuer_master_df","security_identifier_history_df"},
)
security_master_df = b2["security_master_df"]

b4, block_4_manifest = load_manifest_tables(
    BLOCK_4_MANIFEST_PATH,
    {"europe_standard_concept_dictionary_df"},
)

global_canonical_schema_df = (
    b4["europe_standard_concept_dictionary_df"][
        ["standard_concept","statement_type","expected_period_type",
         "expected_unit_family","core_tier","is_core","aggregation_policy"]
    ]
    .drop_duplicates("standard_concept")
    .reset_index(drop=True)
)

print("Canonical concepts:", global_canonical_schema_df["standard_concept"].nunique())

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Canonical concepts: 100


In [ ]:
# 3. KOREAN SECURITY UNIVERSE

def first_col(df, names):
    return next((c for c in names if c in df.columns), None)

def norm_country(v):
    if pd.isna(v): return pd.NA
    t = str(v).strip().upper()
    return {"KOREA":"KR","SOUTH KOREA":"KR","KOR":"KR","KOREA, REPUBLIC OF":"KR"}.get(t,t)

def korean_code(v):
    if pd.isna(v): return pd.NA
    m = re.search(r"(?<!\d)(\d{6})(?!\d)", str(v))
    return m.group(1) if m else pd.NA

country_col = first_col(security_master_df, ["country","issuer_country","domicile_country"])
ticker_col = first_col(security_master_df, ["ticker","primary_ticker","source_ticker"])
name_col = first_col(security_master_df, ["issuer_name","security_name","name"])

korea_security_universe_df = pd.DataFrame(index=security_master_df.index)
korea_security_universe_df["security_id"] = security_master_df.get("security_id")
korea_security_universe_df["issuer_id"] = security_master_df.get("issuer_id")
korea_security_universe_df["issuer_name"] = security_master_df[name_col] if name_col else pd.NA
korea_security_universe_df["ticker"] = security_master_df[ticker_col] if ticker_col else pd.NA
korea_security_universe_df["country"] = (
    security_master_df[country_col].map(norm_country) if country_col else pd.NA
)
korea_security_universe_df["stock_code"] = korea_security_universe_df["ticker"].map(korean_code)

korea_security_universe_df = (
    korea_security_universe_df[
        korea_security_universe_df["country"].eq("KR")
        | korea_security_universe_df["stock_code"].notna()
    ]
    .drop_duplicates()
    .reset_index(drop=True)
)

korea_issuer_universe_df = (
    korea_security_universe_df[["issuer_id","issuer_name","country","stock_code"]]
    .drop_duplicates()
    .reset_index(drop=True)
)

print("Korean securities:", len(korea_security_universe_df))
print("Korean issuers:", len(korea_issuer_universe_df))
display(korea_security_universe_df.head(30))

# ------------------------------------------------
# CURRENT ISSUER-CENTRIC CONTRACTS
# ------------------------------------------------

korea_economic_issuer_universe_df = (
    korea_issuer_universe_df.copy()
)

korea_issuer_security_universe_df = (
    korea_security_universe_df.copy()
)

print(
    "Korean economic issuers:",
    korea_economic_issuer_universe_df[
        "issuer_id"
    ].nunique(),
)

print(
    "Korean issuer securities:",
    len(
        korea_issuer_security_universe_df
    ),
)


Korean securities: 30
Korean issuers: 29


,security_id,issuer_id,issuer_name,ticker,country,stock_code
0,GAS_1923A3C4BFD1F15065D4,GAI_760764847B93BD2F3669,"SKC Co., Ltd.",None,KR,<NA>
1,GAS_2163E459B8C8C0051F63,GAI_DCADE937EAA36FAE9858,"L&F CO., LTD",None,KR,<NA>
2,GAS_253EBF05E7F6109AC51E,GAI_BA351AB2F8C24357BA19,Hyundai Wia Corp,None,KR,<NA>
3,GAS_30CE64375B89BE3EBC21,GAI_2DC02C752B1A08BD8515,HL Mando Corp.,None,KR,<NA>
4,GAS_44190BFA73E43C46F50F,GAI_C35B63ABFCBB61F9132F,Hanon Systems,None,KR,<NA>
5,GAS_468968C965737C0BC664,GAI_84E6DEF2646CE0470492,Chunbo Co Ltd,None,KR,<NA>
6,GAS_4CD743C01EA69D9A78D5,GAI_7CB4C6586DF0E3D9F619,Hyundai Mobis Co Ltd,012330,KR,012330
7,GAS_5FE6D2C688294AF2EAA7,GAI_2BB5AF68192D6F5C8FEB,KCC Corp,None,KR,<NA>
8,GAS_698756D5060CAE43A761,GAI_42D376026BC5F0B1A51E,"COSMO CHEMICAL CO., LTD.",None,KR,<NA>
9,GAS_71BFFA125442D8AB81F4,GAI_D0A4875D18F60203F977,Samsung Electro-Mechanics Co Ltd,None,KR,<NA>


Korean economic issuers: 28
Korean issuer securities: 30


In [ ]:
# 4. OPENDART CLIENT AND CACHE

session = requests.Session()
session.headers.update({
    "User-Agent":"Global Automotive Investment Database research client",
    "Accept":"application/json, application/xml, application/zip",
})

def cache_key(endpoint, params=None):
    payload = json.dumps({"endpoint":endpoint,"params":params or {}}, sort_keys=True, default=str)
    return hashlib.sha256(payload.encode()).hexdigest()

def dart_get_json(endpoint, params, cache_dir, force_refresh=False):
    url = f"{DART_API_BASE}/{endpoint.lstrip('/')}"
    cache_path = cache_dir / f"{cache_key(endpoint, params)}.json"

    if cache_path.exists() and not force_refresh:
        with cache_path.open("r", encoding="utf-8") as f:
            return json.load(f), {"status":"CACHE_HIT","cache_path":str(cache_path),"url":url}

    req = {**params, "crtfc_key":DART_API_KEY}
    last_error = None

    for attempt in range(1, MAX_RETRIES + 1):
        try:
            r = session.get(url, params=req, timeout=REQUEST_TIMEOUT_SECONDS)
            r.raise_for_status()
            payload = r.json()
            status = str(payload.get("status","000"))

            if status not in {"000","013"}:
                raise RuntimeError(f"OpenDART {status}: {payload.get('message')}")

            with cache_path.open("w", encoding="utf-8") as f:
                json.dump(payload, f, ensure_ascii=False)

            time.sleep(REQUEST_INTERVAL_SECONDS)

            return payload, {
                "status":"DOWNLOADED",
                "http_status":r.status_code,
                "api_status":status,
                "api_message":payload.get("message"),
                "cache_path":str(cache_path),
                "url":r.url.replace(DART_API_KEY,"***"),
                "attempt":attempt,
            }
        except Exception as exc:
            last_error = repr(exc)
            time.sleep(min(2 ** attempt, 20))

    raise RuntimeError(f"OpenDART request failed: {endpoint}; {last_error}")

def download_corp_code_zip():
    path = DART_CORP_CACHE_DIR / "corpCode.zip"
    if path.exists():
        return path.read_bytes(), {"status":"CACHE_HIT","cache_path":str(path)}

    r = session.get(
        f"{DART_API_BASE}/corpCode.xml",
        params={"crtfc_key":DART_API_KEY},
        timeout=REQUEST_TIMEOUT_SECONDS,
    )
    r.raise_for_status()

    if not r.content.startswith(b"PK"):
        raise ValueError("corpCode.xml did not return a ZIP archive.")

    path.write_bytes(r.content)
    return r.content, {"status":"DOWNLOADED","http_status":r.status_code,"cache_path":str(path)}

In [ ]:
# 5. CORPORATION-CODE REGISTRY AND SECURITY BRIDGE
# ------------------------------------------------

corp_zip_bytes, corp_code_download_log = (
    download_corp_code_zip()
)

with zipfile.ZipFile(
    BytesIO(corp_zip_bytes)
) as archive:
    member = next(
        name
        for name in archive.namelist()
        if name.lower().endswith(".xml")
    )
    root = ET.fromstring(
        archive.read(member)
    )

rows = []

for item in root.findall(".//list"):
    rows.append({
        child.tag: (
            child.text.strip()
            if child.text
            else pd.NA
        )
        for child in item
    })

dart_corporation_registry_df = (
    pd.DataFrame(rows)
)

for column in [
    "corp_code",
    "stock_code",
]:
    dart_corporation_registry_df[
        column
    ] = (
        dart_corporation_registry_df[
            column
        ]
        .astype("string")
        .str.strip()
    )

dart_listed_corporations_df = (
    dart_corporation_registry_df[
        dart_corporation_registry_df[
            "stock_code"
        ].notna()
        & dart_corporation_registry_df[
            "stock_code"
        ].ne("")
    ]
    .copy()
)

korea_corporation_bridge_df = (
    korea_issuer_security_universe_df
    .merge(
        dart_listed_corporations_df[
            [
                "corp_code",
                "corp_name",
                "corp_eng_name",
                "stock_code",
                "modify_date",
            ]
        ],
        on="stock_code",
        how="left",
        validate="m:m",
    )
)

korea_resolved_corporations_df = (
    korea_corporation_bridge_df[
        korea_corporation_bridge_df[
            "corp_code"
        ].notna()
    ]
    .copy()
    .reset_index(drop=True)
)

korea_unresolved_corporations_df = (
    korea_corporation_bridge_df[
        korea_corporation_bridge_df[
            "corp_code"
        ].isna()
    ]
    .copy()
    .reset_index(drop=True)
)

if MAX_CORPORATIONS is not None:
    korea_resolved_corporations_df = (
        korea_resolved_corporations_df
        .head(
            int(
                MAX_CORPORATIONS
            )
        )
    )


# ------------------------------------------------
# AUTHORITATIVE CORPORATION CODE → ISSUER BRIDGE
# ------------------------------------------------

korea_corp_issuer_bridge_candidates_df = (
    korea_resolved_corporations_df[
        [
            column
            for column in [
                "corp_code",
                "corp_name",
                "corp_eng_name",
                "stock_code",
                "issuer_id",
                "issuer_name",
                "country",
            ]
            if column
            in korea_resolved_corporations_df.columns
        ]
    ]
    .dropna(
        subset=[
            "corp_code",
            "issuer_id",
        ]
    )
    .drop_duplicates()
    .reset_index(drop=True)
)

korea_corp_bridge_quality_df = (
    korea_corp_issuer_bridge_candidates_df
    .groupby(
        "corp_code",
        dropna=False,
    )
    .agg(
        issuer_id_count=(
            "issuer_id",
            "nunique",
        ),
        stock_code_count=(
            "stock_code",
            "nunique",
        ),
        issuer_name_count=(
            "issuer_name",
            "nunique",
        ),
    )
    .reset_index()
)

korea_corp_issuer_bridge_candidates_df = (
    korea_corp_issuer_bridge_candidates_df
    .merge(
        korea_corp_bridge_quality_df,
        on="corp_code",
        how="left",
        validate="m:1",
    )
)

korea_corp_issuer_bridge_df = (
    korea_corp_issuer_bridge_candidates_df[
        korea_corp_issuer_bridge_candidates_df[
            "issuer_id_count"
        ].eq(1)
    ]
    .sort_values(
        [
            "corp_code",
            "issuer_id",
        ],
        na_position="last",
    )
    .drop_duplicates(
        "corp_code",
        keep="first",
    )
    .reset_index(drop=True)
)

korea_corp_issuer_conflicts_df = (
    korea_corp_issuer_bridge_candidates_df[
        ~korea_corp_issuer_bridge_candidates_df[
            "issuer_id_count"
        ].eq(1)
    ]
    .copy()
    .reset_index(drop=True)
)


# ------------------------------------------------
# CORPORATION CODE → SECURITY BRIDGE
# ------------------------------------------------

korea_corp_security_bridge_df = (
    korea_resolved_corporations_df[
        [
            column
            for column in [
                "corp_code",
                "corp_name",
                "stock_code",
                "security_id",
                "issuer_id",
                "issuer_name",
                "ticker",
                "country",
            ]
            if column
            in korea_resolved_corporations_df.columns
        ]
    ]
    .dropna(
        subset=[
            "corp_code",
            "security_id",
            "issuer_id",
        ]
    )
    .drop_duplicates()
    .reset_index(drop=True)
)


# ------------------------------------------------
# PREFERRED SOURCE AND RELATIONSHIP GRAPH
# ------------------------------------------------

korea_preferred_accounting_source_df = (
    korea_corp_issuer_bridge_df[
        [
            column
            for column in [
                "issuer_id",
                "issuer_name",
                "corp_code",
                "stock_code",
                "country",
            ]
            if column
            in korea_corp_issuer_bridge_df.columns
        ]
    ]
    .rename(
        columns={
            "corp_code": (
                "preferred_source_entity_id"
            ),
        }
    )
    .assign(
        preferred_source_system=(
            "OPENDART"
        ),
        preferred_source_region="KOREA",
        source_confidence=1.0,
        selection_basis=(
            "AUTHORITATIVE_CORP_CODE_TO_ISSUER_BRIDGE"
        ),
    )
    .reset_index(drop=True)
)

korea_entity_relationship_graph_df = (
    pd.DataFrame(
        columns=[
            "from_entity_id",
            "to_entity_id",
            "relationship_type",
            "effective_start",
            "effective_end",
            "confidence",
            "source_system",
            "source_region",
        ]
    )
)

print(
    "Resolved security-corporation rows:",
    len(
        korea_resolved_corporations_df
    ),
)

print(
    "Unresolved security-corporation rows:",
    len(
        korea_unresolved_corporations_df
    ),
)

print(
    "Confirmed corporation-code issuer mappings:",
    len(
        korea_corp_issuer_bridge_df
    ),
)

print(
    "Conflicted corporation-code rows:",
    len(
        korea_corp_issuer_conflicts_df
    ),
)

display(
    korea_corp_issuer_bridge_df.head(30)
)

Resolved security-corporation rows: 11
Unresolved security-corporation rows: 19
Confirmed corporation-code issuer mappings: 11
Conflicted corporation-code rows: 0


,corp_code,corp_name,corp_eng_name,stock_code,issuer_id,issuer_name,country,issuer_id_count,stock_code_count,issuer_name_count
0,00106641,기아,KIA CORPORATION,000270,GAI_E0B131AD9F0C6F6AF6FC,Kia Corp,KR,1,1,1
1,00113997,롯데에너지머티리얼즈,LOTTE ENERGY MATERIALS CORPORATION,020150,GAI_2D5FAB8BDC21859A8209,Iljin Materials Co Ltd,KR,1,1,1
2,00126362,삼성SDI,"SAMSUNG SDI CO.,LTD",006400,GAI_9F24E90B90D2C6BA854C,Samsung SDI Co Ltd,KR,1,1,1
3,00126380,삼성전자,"SAMSUNG ELECTRONICS CO,.LTD",005930,GAI_1E86434257B4B73EA9E3,Samsung Electronics Co Ltd,KR,1,1,1
4,00164742,현대자동차,HYUNDAI MOTOR CO,005380,GAI_5CC40356A3DB397D5423,Hyundai Motor Co,KR,1,1,1
5,00164788,현대모비스,"HYUNDAI MOBIS CO.,LTD",012330,GAI_7CB4C6586DF0E3D9F619,Hyundai Mobis Co Ltd,KR,1,1,1
6,00356361,LG화학,LG CHEM LTD,051910,GAI_DFAC684BA54575ACF490,LG Chem Ltd,KR,1,1,1
7,00525934,LX세미콘,"LX Semicon Co., Ltd.",108320,GAI_FEB9CC791DD64E319F98,LX Semicon Co Ltd,KR,1,1,1
8,01160363,에코프로비엠,"ECOPRO BM CO.,LTD.",247540,GAI_AADC3296814408EEED3E,"ECOPRO BM CO.,LTD.",KR,1,1,1
9,01386916,SK아이이테크놀로지,"SK ie technology Co., Ltd.",361610,GAI_CEA6FB761B39BA3C116A,SK IE Technology Co Ltd,KR,1,1,1


In [ ]:
# 6. FILING HISTORIES
# ------------------------------------------------

def dart_date(value):
    return pd.Timestamp(
        value
    ).strftime("%Y%m%d")


def fetch_history(corp_code):
    rows = []
    logs = []
    page_no = 1

    while True:
        params = {
            "corp_code": corp_code,
            "bgn_de": dart_date(
                DISCOVERY_START_DATE
            ),
            "end_de": dart_date(
                DISCOVERY_END_DATE
            ),
            "pblntf_ty": "A",
            "page_no": page_no,
            "page_count": 100,
        }

        payload, log = dart_get_json(
            "list.json",
            params,
            DART_FILING_CACHE_DIR,
        )

        log.update({
            "corp_code": corp_code,
            "page_no": page_no,
        })

        logs.append(log)

        rows.extend(
            payload.get(
                "list",
                [],
            )
            or []
        )

        if page_no >= int(
            payload.get(
                "total_page",
                1,
            )
            or 1
        ):
            break

        page_no += 1

    return (
        pd.DataFrame(rows),
        pd.DataFrame(logs),
    )


filing_frames = []
log_frames = []

corp_codes = (
    korea_corp_issuer_bridge_df[
        "corp_code"
    ]
    .dropna()
    .drop_duplicates()
    .astype(str)
    .tolist()
)

for corp_code in tqdm(
    corp_codes,
    desc="DART filing histories",
):
    filing_frame, log_frame = (
        fetch_history(
            corp_code
        )
    )

    if not filing_frame.empty:
        filing_frames.append(
            filing_frame
        )

    if not log_frame.empty:
        log_frames.append(
            log_frame
        )

dart_filings_raw_df = (
    pd.concat(
        filing_frames,
        ignore_index=True,
    )
    if filing_frames
    else pd.DataFrame()
)

dart_filing_download_log_df = (
    pd.concat(
        log_frames,
        ignore_index=True,
    )
    if log_frames
    else pd.DataFrame()
)


if dart_filings_raw_df.empty:
    korea_filing_metadata_df = (
        pd.DataFrame()
    )
    korea_matched_filings_df = (
        pd.DataFrame()
    )
    korea_unmatched_filings_df = (
        pd.DataFrame()
    )

else:
    metadata = (
        dart_filings_raw_df
        .copy()
    )

    metadata[
        "filing_id"
    ] = metadata[
        "rcept_no"
    ].astype("string")

    metadata[
        "receipt_number"
    ] = metadata[
        "rcept_no"
    ].astype("string")

    metadata[
        "receipt_date"
    ] = pd.to_datetime(
        metadata[
            "rcept_dt"
        ],
        format="%Y%m%d",
        errors="coerce",
    )

    metadata[
        "available_datetime"
    ] = (
        metadata[
            "receipt_date"
        ]
        .dt.tz_localize(
            "Asia/Seoul"
        )
        .dt.tz_convert(
            "UTC"
        )
    )

    metadata[
        "available_date"
    ] = (
        metadata[
            "available_datetime"
        ]
        .dt.normalize()
    )

    metadata[
        "is_amendment"
    ] = (
        metadata[
            "report_nm"
        ]
        .astype("string")
        .str.contains(
            r"기재정정|첨부정정|정정",
            na=False,
        )
    )

    metadata[
        "availability_basis"
    ] = "DART_RECEIPT_DATE"

    metadata[
        "source_system"
    ] = "OPENDART"

    issuer_bridge = (
        korea_corp_issuer_bridge_df[
            [
                column
                for column in [
                    "corp_code",
                    "corp_name",
                    "corp_eng_name",
                    "stock_code",
                    "issuer_id",
                    "issuer_name",
                    "country",
                ]
                if column
                in korea_corp_issuer_bridge_df.columns
            ]
        ]
        .drop_duplicates(
            "corp_code"
        )
    )

    korea_filing_metadata_df = (
        metadata
        .merge(
            issuer_bridge,
            on="corp_code",
            how="left",
            validate="m:1",
        )
        .drop_duplicates(
            [
                "rcept_no",
                "issuer_id",
            ]
        )
        .reset_index(drop=True)
    )

    korea_filing_metadata_df[
        "issuer_link_status"
    ] = np.where(
        korea_filing_metadata_df[
            "issuer_id"
        ].notna(),
        "LINKED",
        "UNRESOLVED_CORP_CODE_TO_ISSUER",
    )

    korea_matched_filings_df = (
        korea_filing_metadata_df[
            korea_filing_metadata_df[
                "issuer_id"
            ].notna()
        ]
        .copy()
    )

    korea_unmatched_filings_df = (
        korea_filing_metadata_df[
            korea_filing_metadata_df[
                "issuer_id"
            ].isna()
        ]
        .copy()
    )


print(
    "Issuer-level filing metadata rows:",
    len(
        korea_filing_metadata_df
    ),
)

print(
    "Matched issuer-level filings:",
    len(
        korea_matched_filings_df
    ),
)

print(
    "Distinct filing IDs:",
    (
        korea_filing_metadata_df[
            "filing_id"
        ].nunique()
        if not korea_filing_metadata_df.empty
        else 0
    ),
)

DART filing histories:   0%|          | 0/11 [00:00<?, ?it/s]

Issuer-level filing metadata rows: 322
Matched issuer-level filings: 322
Distinct filing IDs: 322


In [ ]:
# 7. FINANCIAL REQUEST GRID

years = range(pd.Timestamp(DISCOVERY_START_DATE).year, pd.Timestamp(DISCOVERY_END_DATE).year + 1)

request_rows = [
    {
        "corp_code":corp,
        "business_year":year,
        "report_code":report_code,
        "report_period":report_period,
        "fs_div":fs_div,
        "fs_scope":fs_scope,
    }
    for corp in corp_codes
    for year in years
    for report_code, report_period in REPORT_CODES.items()
    for fs_div, fs_scope in FS_DIVISIONS.items()
]

dart_financial_request_grid_df = pd.DataFrame(request_rows)

if MAX_FINANCIAL_REQUESTS is not None:
    dart_financial_request_grid_df = dart_financial_request_grid_df.head(
        int(MAX_FINANCIAL_REQUESTS)
    )

print("Financial requests:", len(dart_financial_request_grid_df))

Financial requests: 704


In [ ]:
# 8. RETRIEVE FULL FINANCIAL ACCOUNTS

financial_frames, financial_logs = [], []

for row in tqdm(
    dart_financial_request_grid_df.itertuples(index=False),
    total=len(dart_financial_request_grid_df),
    desc="DART financial accounts",
):
    params = {
        "corp_code":str(row.corp_code),
        "bsns_year":str(row.business_year),
        "reprt_code":str(row.report_code),
        "fs_div":str(row.fs_div),
    }

    try:
        payload, log = dart_get_json(
            "fnlttSinglAcntAll.json",
            params,
            DART_FINANCIAL_CACHE_DIR,
        )
        result = payload.get("list",[]) or []

        if result:
            frame = pd.DataFrame(result)
            frame["requested_business_year"] = row.business_year
            frame["requested_report_code"] = row.report_code
            frame["requested_report_period"] = row.report_period
            frame["requested_fs_div"] = row.fs_div
            frame["requested_fs_scope"] = row.fs_scope
            financial_frames.append(frame)

        log.update({
            "corp_code":row.corp_code,
            "business_year":row.business_year,
            "report_code":row.report_code,
            "fs_div":row.fs_div,
            "result_count":len(result),
        })
        financial_logs.append(log)

    except Exception as exc:
        financial_logs.append({
            "corp_code":row.corp_code,
            "business_year":row.business_year,
            "report_code":row.report_code,
            "fs_div":row.fs_div,
            "status":"FAILED",
            "result_count":0,
            "error":repr(exc),
        })

dart_financial_accounts_raw_df = (
    pd.concat(financial_frames, ignore_index=True) if financial_frames else pd.DataFrame()
)
dart_financial_download_log_df = pd.DataFrame(financial_logs)

print("Raw financial rows:", len(dart_financial_accounts_raw_df))

DART financial accounts:   0%|          | 0/704 [00:00<?, ?it/s]

Raw financial rows: 92634


In [ ]:
# 9. NORMALISE VALUES AND ATTACH RECEIPT-LEVEL AVAILABILITY
# ------------------------------------------------

def num(value):
    if pd.isna(value):
        return np.nan

    text = (
        str(value)
        .replace(
            ",",
            "",
        )
        .replace(
            "△",
            "-",
        )
        .replace(
            "▲",
            "-",
        )
        .strip()
    )

    return pd.to_numeric(
        text,
        errors="coerce",
    )


def period_end(
    year,
    code,
):
    suffix = {
        "11011": "12-31",
        "11012": "06-30",
        "11013": "03-31",
        "11014": "09-30",
    }[
        str(code)
    ]

    return pd.Timestamp(
        f"{int(year)}-{suffix}"
    )


def report_pattern(code):
    return {
        "11011": re.compile(
            r"사업보고서|annual",
            re.I,
        ),
        "11012": re.compile(
            r"반기보고서|semi",
            re.I,
        ),
        "11013": re.compile(
            r"분기보고서.*1분기|quarter.*1",
            re.I,
        ),
        "11014": re.compile(
            r"분기보고서.*3분기|quarter.*3",
            re.I,
        ),
    }.get(
        str(code)
    )


if dart_financial_accounts_raw_df.empty:
    dart_financial_receipt_bridge_df = (
        pd.DataFrame()
    )
    korea_financial_facts_pit_df = (
        pd.DataFrame()
    )

else:
    facts = (
        dart_financial_accounts_raw_df
        .copy()
    )

    facts[
        "reported_value"
    ] = facts[
        "thstrm_amount"
    ].map(num)

    facts[
        "prior_year_value"
    ] = facts.get(
        "frmtrm_amount",
        pd.Series(
            index=facts.index,
            dtype="object",
        ),
    ).map(num)

    facts[
        "two_year_prior_value"
    ] = facts.get(
        "bfefrmtrm_amount",
        pd.Series(
            index=facts.index,
            dtype="object",
        ),
    ).map(num)

    facts[
        "period_end"
    ] = facts.apply(
        lambda row: period_end(
            row[
                "requested_business_year"
            ],
            row[
                "requested_report_code"
            ],
        ),
        axis=1,
    )

    facts[
        "period_type"
    ] = np.where(
        facts[
            "sj_div"
        ].eq("BS"),
        "INSTANT",
        "DURATION",
    )

    facts[
        "observed_unit_family"
    ] = "MONETARY"

    facts[
        "unit"
    ] = "KRW"

    facts[
        "account_id_clean"
    ] = (
        facts[
            "account_id"
        ]
        .astype("string")
        .str.strip()
    )

    facts[
        "account_name_clean"
    ] = (
        facts[
            "account_nm"
        ]
        .astype("string")
        .str.replace(
            r"\s+",
            " ",
            regex=True,
        )
        .str.strip()
    )

    facts[
        "fact_key"
    ] = (
        facts[
            [
                "corp_code",
                "requested_business_year",
                "requested_report_code",
                "requested_fs_div",
                "sj_div",
                "account_id_clean",
                "ord",
            ]
        ]
        .astype("string")
        .agg(
            "|".join,
            axis=1,
        )
    )

    receipt_rows = []

    for row in facts[
        [
            "corp_code",
            "requested_business_year",
            "requested_report_code",
            "period_end",
        ]
    ].drop_duplicates().itertuples(
        index=False
    ):
        candidates = (
            korea_filing_metadata_df[
                korea_filing_metadata_df[
                    "corp_code"
                ].eq(
                    row.corp_code
                )
            ]
            .copy()
        )

        pattern = report_pattern(
            row.requested_report_code
        )

        if pattern is not None:
            candidates = candidates[
                candidates[
                    "report_nm"
                ]
                .astype("string")
                .str.contains(
                    pattern,
                    na=False,
                )
            ]

        candidates[
            "date_distance"
        ] = (
            candidates[
                "receipt_date"
            ]
            - row.period_end
        ).dt.days.abs()

        candidates = (
            candidates
            .sort_values(
                [
                    "is_amendment",
                    "date_distance",
                    "receipt_date",
                ],
                ascending=[
                    True,
                    True,
                    True,
                ],
            )
        )

        if not candidates.empty:
            selected = (
                candidates.iloc[0]
            )

            receipt_rows.append({
                "corp_code": row.corp_code,
                "requested_business_year": row.requested_business_year,
                "requested_report_code": row.requested_report_code,
                "rcept_no": selected.get(
                    "rcept_no"
                ),
                "filing_id": selected.get(
                    "filing_id"
                ),
                "receipt_number": selected.get(
                    "receipt_number"
                ),
                "report_nm": selected.get(
                    "report_nm"
                ),
                "receipt_date": selected.get(
                    "receipt_date"
                ),
                "available_datetime": selected.get(
                    "available_datetime"
                ),
                "available_date": selected.get(
                    "available_date"
                ),
                "availability_basis": selected.get(
                    "availability_basis"
                ),
            })

    dart_financial_receipt_bridge_df = (
        pd.DataFrame(
            receipt_rows
        )
    )

    facts = facts.merge(
        dart_financial_receipt_bridge_df,
        on=[
            "corp_code",
            "requested_business_year",
            "requested_report_code",
        ],
        how="left",
        validate="m:1",
    )

    receipt_candidates = [
        column
        for column in [
            "rcept_no",
            "rcept_no_y",
            "rcept_no_x",
        ]
        if column
        in facts.columns
    ]

    if receipt_candidates:
        facts[
            "rcept_no"
        ] = (
            facts[
                receipt_candidates
            ]
            .bfill(axis=1)
            .iloc[:, 0]
            .astype("string")
        )

        facts = facts.drop(
            columns=[
                column
                for column in [
                    "rcept_no_x",
                    "rcept_no_y",
                ]
                if column
                in facts.columns
            ]
        )

    else:
        facts[
            "rcept_no"
        ] = pd.NA

    facts[
        "filing_id"
    ] = (
        facts.get(
            "filing_id",
            pd.Series(
                pd.NA,
                index=facts.index,
                dtype="string",
            ),
        )
        .astype("string")
        .fillna(
            facts[
                "rcept_no"
            ]
        )
    )

    facts[
        "receipt_number"
    ] = (
        facts.get(
            "receipt_number",
            pd.Series(
                pd.NA,
                index=facts.index,
                dtype="string",
            ),
        )
        .astype("string")
        .fillna(
            facts[
                "rcept_no"
            ]
        )
    )

    issuer_bridge = (
        korea_corp_issuer_bridge_df[
            [
                column
                for column in [
                    "corp_code",
                    "corp_name",
                    "stock_code",
                    "issuer_id",
                    "issuer_name",
                    "country",
                ]
                if column
                in korea_corp_issuer_bridge_df.columns
            ]
        ]
        .drop_duplicates(
            "corp_code"
        )
    )

    korea_financial_facts_pit_df = (
        facts
        .merge(
            issuer_bridge,
            on="corp_code",
            how="left",
            validate="m:1",
        )
    )

    korea_financial_facts_pit_df[
        "issuer_link_status"
    ] = np.where(
        korea_financial_facts_pit_df[
            "issuer_id"
        ].notna(),
        "LINKED",
        "UNRESOLVED_CORP_CODE_TO_ISSUER",
    )


print(
    "Issuer-level point-in-time Korean facts:",
    len(
        korea_financial_facts_pit_df
    ),
)

print(
    "Point-in-time facts with issuer IDs:",
    (
        int(
            korea_financial_facts_pit_df[
                "issuer_id"
            ].notna().sum()
        )
        if not korea_financial_facts_pit_df.empty
        else 0
    ),
)

print(
    "Distinct filing IDs in PIT facts:",
    (
        korea_financial_facts_pit_df[
            "filing_id"
        ].nunique()
        if not korea_financial_facts_pit_df.empty
        else 0
    ),
)

Issuer-level point-in-time Korean facts: 92634
Point-in-time facts with issuer IDs: 92634
Distinct filing IDs in PIT facts: 294


In [ ]:
# 10. KOREAN ACCOUNT MAPPINGS

# standard_concept, account-id regex, Korean/English account-name regex, priority
MAP = [
("revenue",r"(Revenue|SalesRevenue|OperatingRevenue)$",r"매출액|영업수익|수익",1),
("cost_of_revenue",r"(CostOfSales|CostOfRevenue)$",r"매출원가",1),
("gross_profit",r"GrossProfit$",r"매출총이익",1),
("operating_income",r"(OperatingIncomeLoss|OperatingProfitLoss)$",r"영업이익|영업손실",1),
("profit_before_tax",r"(ProfitLossBeforeTax|IncomeLossBeforeTax)$",r"법인세비용차감전",1),
("income_tax_expense",r"(IncomeTaxExpense|IncomeTaxExpenseBenefit)$",r"법인세비용",1),
("net_income",r"(ProfitLoss|NetIncomeLoss)$",r"당기순이익|당기순손실",1),
("net_income_attributable_to_owners",r"ProfitLossAttributableToOwnersOfParent$",r"지배기업.*소유주.*당기순",1),
("basic_eps",r"BasicEarningsLossPerShare$",r"기본주당이익",1),
("diluted_eps",r"DilutedEarningsLossPerShare$",r"희석주당이익",1),
("research_and_development_expense",r"ResearchAndDevelopmentExpense$",r"연구개발비",1),
("selling_general_and_administrative_expense",r"SellingGeneralAndAdministrativeExpense$",r"판매비와관리비",1),
("employee_benefit_expense",r"EmployeeBenefitsExpense$",r"종업원급여",1),
("depreciation_expense",r"DepreciationExpense$",r"감가상각비",1),
("amortisation_expense",r"AmortisationExpense$",r"무형자산상각비",1),
("impairment_loss",r"ImpairmentLoss$",r"손상차손",1),
("interest_expense",r"InterestExpense$",r"이자비용",1),
("interest_income",r"InterestIncome$",r"이자수익",1),
("other_comprehensive_income",r"OtherComprehensiveIncome$",r"기타포괄손익",1),
("comprehensive_income",r"ComprehensiveIncome$",r"총포괄손익",1),

("total_assets",r"Assets$",r"자산총계",1),
("current_assets",r"CurrentAssets$",r"유동자산",1),
("noncurrent_assets",r"NoncurrentAssets$",r"비유동자산",1),
("cash_and_cash_equivalents",r"CashAndCashEquivalents$",r"현금및현금성자산",1),
("trade_receivables",r"TradeReceivables$",r"매출채권",1),
("other_receivables",r"OtherReceivables$",r"기타채권|미수금",1),
("inventory",r"Inventories$",r"재고자산",1),
("raw_material_inventory",r"RawMaterialsAndSupplies$",r"원재료",1),
("work_in_progress_inventory",r"WorkInProgress$",r"재공품",1),
("finished_goods_inventory",r"FinishedGoods$",r"제품|상품",1),
("property_plant_equipment",r"PropertyPlantAndEquipment$",r"유형자산",1),
("right_of_use_assets",r"RightofuseAssets$",r"사용권자산",1),
("goodwill",r"Goodwill$",r"영업권",1),
("intangible_assets",r"IntangibleAssetsOtherThanGoodwill$",r"무형자산",1),
("capitalised_development_costs",r"DevelopmentCosts$",r"개발비",1),
("deferred_tax_assets",r"DeferredTaxAssets$",r"이연법인세자산",1),

("total_liabilities",r"Liabilities$",r"부채총계",1),
("current_liabilities",r"CurrentLiabilities$",r"유동부채",1),
("noncurrent_liabilities",r"NoncurrentLiabilities$",r"비유동부채",1),
("trade_payables",r"TradePayables$",r"매입채무",1),
("other_payables",r"OtherPayables$",r"미지급금|기타채무",1),
("contract_liabilities",r"ContractLiabilities$",r"계약부채",1),
("short_term_debt",r"(CurrentBorrowings|ShorttermBorrowings|CurrentPortionOfLongtermBorrowings)$",r"단기차입금|유동성장기부채",1),
("long_term_debt",r"(NoncurrentBorrowings|LongtermBorrowings)$",r"장기차입금|사채",1),
("total_borrowings",r"Borrowings$",r"이자부부채|차입금.*총계",1),
("current_lease_liabilities",r"CurrentLeaseLiabilities$",r"유동.*리스부채",1),
("noncurrent_lease_liabilities",r"NoncurrentLeaseLiabilities$",r"비유동.*리스부채",1),
("warranty_provisions",r"WarrantyProvisions$",r"판매보증충당부채|제품보증",1),
("pension_liabilities",r"NetDefinedBenefitLiability$",r"순확정급여부채|퇴직급여",1),
("deferred_tax_liabilities",r"DeferredTaxLiabilities$",r"이연법인세부채",1),

("total_equity",r"Equity$",r"자본총계",1),
("equity_attributable_to_owners",r"EquityAttributableToOwnersOfParent$",r"지배기업.*소유주.*지분",1),
("noncontrolling_interests",r"NoncontrollingInterests$",r"비지배지분",1),
("share_capital",r"IssuedCapital$",r"자본금",1),
("share_premium",r"SharePremium$",r"주식발행초과금|자본잉여금",1),
("retained_earnings",r"RetainedEarnings$",r"이익잉여금",1),
("treasury_shares",r"TreasuryShares$",r"자기주식",1),

("operating_cash_flow",r"CashFlowsFromUsedInOperatingActivities$",r"영업활동.*현금흐름",1),
("investing_cash_flow",r"CashFlowsFromUsedInInvestingActivities$",r"투자활동.*현금흐름",1),
("financing_cash_flow",r"CashFlowsFromUsedInFinancingActivities$",r"재무활동.*현금흐름",1),
("capital_expenditure",r"(PurchaseOfPropertyPlantAndEquipment|PaymentsToAcquirePropertyPlantAndEquipment)$",r"유형자산.*취득",1),
("intangible_asset_purchases",r"PurchaseOfIntangibleAssets$",r"무형자산.*취득",1),
("debt_issuance",r"ProceedsFromBorrowings$",r"차입.*증가|사채.*발행",1),
("debt_repayment",r"RepaymentsOfBorrowings$",r"차입금.*상환|사채.*상환",1),
("dividends_paid",r"DividendsPaid$",r"배당금.*지급",1),
("share_repurchases",r"PaymentsToAcquireOrRedeemEntitysShares$",r"자기주식.*취득",1),
("interest_paid",r"InterestPaid$",r"이자.*지급",1),
("interest_received",r"InterestReceived$",r"이자.*수취",1),
("income_taxes_paid",r"IncomeTaxesPaid$",r"법인세.*납부",1),
("cash_change",r"IncreaseDecreaseInCashAndCashEquivalents$",r"현금및현금성자산.*증감",1),
]

korea_source_account_mapping_df = pd.DataFrame(
    MAP,
    columns=["standard_concept","account_id_regex","account_name_regex","priority"],
)

korea_standard_concept_dictionary_df = korea_source_account_mapping_df.merge(
    global_canonical_schema_df,
    on="standard_concept",
    how="left",
    validate="m:1",
)

unknown = korea_standard_concept_dictionary_df[
    korea_standard_concept_dictionary_df["statement_type"].isna()
]["standard_concept"].drop_duplicates().tolist()

if unknown:
    raise RuntimeError(f"Unknown canonical concepts: {unknown}")

print("Mapping rows:", len(korea_standard_concept_dictionary_df))
print("Canonical concepts represented:", korea_standard_concept_dictionary_df["standard_concept"].nunique())

Mapping rows: 70
Canonical concepts represented: 70


In [ ]:
# 11. MAP ACCOUNTS, RESOLVE DUPLICATES AND INVENTORY EXTENSIONS

def candidates(account_id, account_name):
    d = korea_standard_concept_dictionary_df

    id_mask = d["account_id_regex"].map(
        lambda p: bool(re.search(p, str(account_id), flags=re.I))
    )
    name_mask = d["account_name_regex"].map(
        lambda p: bool(re.search(p, str(account_name), flags=re.I))
    )

    out = d[id_mask | name_mask].copy()
    out["matched_by_account_id"] = id_mask[id_mask | name_mask].to_numpy()
    out["matched_by_account_name"] = name_mask[id_mask | name_mask].to_numpy()
    return out

if korea_financial_facts_pit_df.empty:
    korea_observed_account_mapping_df = pd.DataFrame()
    korea_fundamentals_mapped_df = pd.DataFrame()
    korea_fundamentals_standardised_df = pd.DataFrame()
    korea_unmapped_account_inventory_df = pd.DataFrame()
    korea_automotive_extension_candidates_df = pd.DataFrame()
else:
    mapping_frames = []

    for r in korea_financial_facts_pit_df[
        ["account_id_clean","account_name_clean"]
    ].drop_duplicates().itertuples(index=False):

        c = candidates(r.account_id_clean, r.account_name_clean)

        if not c.empty:
            c["account_id_clean"] = r.account_id_clean
            c["account_name_clean"] = r.account_name_clean
            mapping_frames.append(c)

    korea_observed_account_mapping_df = (
        pd.concat(mapping_frames, ignore_index=True)
        if mapping_frames else pd.DataFrame()
    )

    mapped = korea_financial_facts_pit_df.merge(
        korea_observed_account_mapping_df,
        on=["account_id_clean","account_name_clean"],
        how="left",
        validate="m:m",
    )

    korea_fundamentals_mapped_df = mapped[
        mapped["standard_concept"].notna()
    ].copy()

    korea_fundamentals_mapped_df["period_type_match"] = (
        korea_fundamentals_mapped_df["period_type"]
        == korea_fundamentals_mapped_df["expected_period_type"]
    )
    korea_fundamentals_mapped_df["unit_family_match"] = (
        korea_fundamentals_mapped_df["observed_unit_family"]
        == korea_fundamentals_mapped_df["expected_unit_family"]
    )
    korea_fundamentals_mapped_df["is_numeric_fact"] = (
        korea_fundamentals_mapped_df["reported_value"].notna()
    )

    korea_fundamentals_mapped_df["selection_score"] = (
        korea_fundamentals_mapped_df["priority"] * 100
        + (~korea_fundamentals_mapped_df["matched_by_account_id"]) * 25
        + (~korea_fundamentals_mapped_df["period_type_match"]) * 20
        + (~korea_fundamentals_mapped_df["unit_family_match"]) * 10
        + (~korea_fundamentals_mapped_df["is_numeric_fact"]) * 5
    )

    key = [
        "issuer_id",
        "standard_concept",
        "requested_business_year",
        "requested_report_code",
        "requested_fs_div",
        "period_end",
        "available_datetime",
        "unit",
    ]

    korea_fundamentals_mapped_df = (
        korea_fundamentals_mapped_df
        .sort_values(key + ["selection_score","ord","fact_key"])
        .reset_index(drop=True)
    )

    korea_fundamentals_mapped_df["concept_selection_rank"] = (
        korea_fundamentals_mapped_df.groupby(key, dropna=False).cumcount() + 1
    )
    korea_fundamentals_mapped_df["is_selected_standard_fact"] = (
        korea_fundamentals_mapped_df["concept_selection_rank"].eq(1)
    )

    korea_fundamentals_standardised_df = (
        korea_fundamentals_mapped_df[
            korea_fundamentals_mapped_df["is_selected_standard_fact"]
        ]
        .copy()
        .reset_index(drop=True)
    )

    mapped_pairs = set(zip(
        korea_observed_account_mapping_df["account_id_clean"].astype("string"),
        korea_observed_account_mapping_df["account_name_clean"].astype("string"),
    ))

    receipt_number_candidates = [
        column
        for column in [
            "rcept_no",
            "rcept_no_y",
            "rcept_no_x",
        ]
        if column in korea_financial_facts_pit_df.columns
    ]

    if receipt_number_candidates:

        korea_financial_facts_pit_df["rcept_no_canonical"] = (
            korea_financial_facts_pit_df[
                receipt_number_candidates
            ]
            .bfill(axis=1)
            .iloc[:, 0]
            .astype("string")
        )

    else:

        korea_financial_facts_pit_df["rcept_no_canonical"] = pd.NA


    raw_accounts = korea_financial_facts_pit_df[
        [
            "account_id_clean",
            "account_name_clean",
            "sj_div",
            "requested_fs_div",
            "reported_value",
            "corp_code",
            "rcept_no_canonical",
            "available_datetime",
        ]
    ].copy()

    raw_accounts = raw_accounts.rename(
        columns={
            "rcept_no_canonical": "rcept_no",
        }
    )

    raw_accounts["is_mapped"] = [
        (str(a),str(n)) in mapped_pairs
        for a,n in zip(raw_accounts["account_id_clean"], raw_accounts["account_name_clean"])
    ]

    unmapped = raw_accounts[~raw_accounts["is_mapped"]].copy()

    korea_unmapped_account_inventory_df = (
        unmapped
        .groupby(
            ["account_id_clean","account_name_clean","sj_div","requested_fs_div"],
            dropna=False,
        )
        .agg(
            fact_rows=("reported_value","size"),
            issuer_count=("corp_code","nunique"),
            filing_count=("rcept_no","nunique"),
            numeric_fact_share=("reported_value",lambda s:s.notna().mean()),
            earliest_available=("available_datetime","min"),
            latest_available=("available_datetime","max"),
        )
        .reset_index()
        .sort_values(["issuer_count","fact_rows"],ascending=False)
        .reset_index(drop=True)
    )

    auto_pattern = re.compile(
        r"(vehicle|automotive|production|deliver|warranty|dealer|battery|electric|"
        r"자동차|차량|생산|판매대수|보증|배터리|전기차)",
        re.I,
    )

    text = (
        korea_unmapped_account_inventory_df["account_id_clean"].astype("string")
        + " "
        + korea_unmapped_account_inventory_df["account_name_clean"].astype("string")
    )

    korea_automotive_extension_candidates_df = (
        korea_unmapped_account_inventory_df[text.str.contains(auto_pattern,na=False)]
        .copy()
        .reset_index(drop=True)
    )

print("Mapped candidates:", len(korea_fundamentals_mapped_df))
print("Selected facts:", len(korea_fundamentals_standardised_df))
print("Unmapped accounts:", len(korea_unmapped_account_inventory_df))
print("Automotive extensions:", len(korea_automotive_extension_candidates_df))


# ------------------------------------------------
# SEPARATE SECURITY-EXPANDED FACT TABLE
# ------------------------------------------------

def attach_korean_security_ids(
    issuer_level_facts: pd.DataFrame,
    security_bridge: pd.DataFrame,
) -> pd.DataFrame:

    if issuer_level_facts.empty:
        return issuer_level_facts.copy()

    bridge = (
        security_bridge[
            [
                column
                for column in [
                    "corp_code",
                    "stock_code",
                    "security_id",
                    "issuer_id",
                    "ticker",
                    "country",
                ]
                if column
                in security_bridge.columns
            ]
        ]
        .dropna(
            subset=[
                "issuer_id",
                "security_id",
            ]
        )
        .drop_duplicates()
    )

    join_columns = [
        column
        for column in [
            "issuer_id",
            "corp_code",
        ]
        if (
            column
            in issuer_level_facts.columns
            and column
            in bridge.columns
        )
    ]

    if "issuer_id" not in join_columns:
        raise KeyError(
            "issuer_id is required for "
            "Korean security expansion."
        )

    return issuer_level_facts.merge(
        bridge,
        on=join_columns,
        how="left",
        suffixes=(
            "",
            "_security",
        ),
        validate="m:m",
    )


korea_fundamentals_security_linked_df = (
    attach_korean_security_ids(
        korea_fundamentals_standardised_df,
        korea_corp_security_bridge_df,
    )
)

issuer_level_rows = len(
    korea_fundamentals_standardised_df
)

security_expanded_rows = len(
    korea_fundamentals_security_linked_df
)

korea_issuer_security_link_quality_df = (
    pd.DataFrame({
        "metric": [
            "issuer_level_fact_rows",
            "security_expanded_fact_rows",
            "row_multiplication_ratio",
            "issuer_level_issuers",
            "security_expanded_issuers",
            "linked_security_count",
            "security_linked_row_share",
            "rows_without_security_id",
        ],
        "value": [
            issuer_level_rows,
            security_expanded_rows,
            (
                security_expanded_rows
                / issuer_level_rows
                if issuer_level_rows > 0
                else np.nan
            ),
            (
                korea_fundamentals_standardised_df[
                    "issuer_id"
                ].nunique()
                if not korea_fundamentals_standardised_df.empty
                else 0
            ),
            (
                korea_fundamentals_security_linked_df[
                    "issuer_id"
                ].nunique()
                if not korea_fundamentals_security_linked_df.empty
                else 0
            ),
            (
                korea_fundamentals_security_linked_df[
                    "security_id"
                ].nunique()
                if (
                    not korea_fundamentals_security_linked_df.empty
                    and "security_id"
                    in korea_fundamentals_security_linked_df.columns
                )
                else 0
            ),
            (
                korea_fundamentals_security_linked_df[
                    "security_id"
                ].notna().mean()
                if (
                    not korea_fundamentals_security_linked_df.empty
                    and "security_id"
                    in korea_fundamentals_security_linked_df.columns
                )
                else np.nan
            ),
            (
                int(
                    korea_fundamentals_security_linked_df[
                        "security_id"
                    ].isna().sum()
                )
                if (
                    not korea_fundamentals_security_linked_df.empty
                    and "security_id"
                    in korea_fundamentals_security_linked_df.columns
                )
                else 0
            ),
        ],
    })
)

print(
    "Security-expanded Korean facts:",
    f"{security_expanded_rows:,}",
)

display(
    korea_issuer_security_link_quality_df
)


/tmp/ipykernel_1361/533146510.py:195: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  korea_unmapped_account_inventory_df[text.str.contains(auto_pattern,na=False)]


Mapped candidates: 80883
Selected facts: 29678
Unmapped accounts: 1893
Automotive extensions: 72
Security-expanded Korean facts: 29,678


,metric,value
0,issuer_level_fact_rows,29678.0
1,security_expanded_fact_rows,29678.0
2,row_multiplication_ratio,1.0
3,issuer_level_issuers,11.0
4,security_expanded_issuers,11.0
5,linked_security_count,11.0
6,security_linked_row_share,1.0
7,rows_without_security_id,0.0


In [ ]:
# 12. POINT-IN-TIME HELPERS

def korean_fundamentals_as_of(
    dataframe,
    as_of_date,
    issuer_ids=None,
    security_ids=None,
    standard_concepts=None,
):
    if dataframe.empty:
        return dataframe.copy()

    cutoff = pd.Timestamp(as_of_date)
    cutoff = cutoff.tz_localize("UTC") if cutoff.tzinfo is None else cutoff.tz_convert("UTC")

    result = dataframe[
        pd.to_datetime(dataframe["available_datetime"], errors="coerce", utc=True) <= cutoff
    ].copy()

    if issuer_ids is not None:
        result = result[result["issuer_id"].isin(set(issuer_ids))]
    if security_ids is not None:
        result = result[result["security_id"].isin(set(security_ids))]
    if standard_concepts is not None:
        result = result[result["standard_concept"].isin(set(standard_concepts))]

    return result

def latest_korean_fact_as_of(dataframe, as_of_date):
    result = korean_fundamentals_as_of(dataframe, as_of_date)
    if result.empty:
        return result

    return (
        result
        .sort_values(["period_end","available_datetime"])
        .drop_duplicates(
            ["security_id","issuer_id","standard_concept"],
            keep="last",
        )
        .reset_index(drop=True)
    )

In [ ]:
# 13. COVERAGE, AVAILABILITY AND QUALITY REPORTS

korea_filing_coverage_report_df = pd.DataFrame({
    "metric":[
        "korean_securities","korean_issuers","resolved_dart_corporations",
        "filing_metadata_rows","raw_financial_account_rows",
        "point_in_time_financial_fact_rows","standardised_fact_rows",
    ],
    "value":[
        len(korea_security_universe_df),
        korea_issuer_universe_df["issuer_id"].nunique(),
        korea_resolved_corporations_df["corp_code"].nunique(),
        len(korea_filing_metadata_df),
        len(dart_financial_accounts_raw_df),
        len(korea_financial_facts_pit_df),
        len(korea_fundamentals_standardised_df),
    ],
})

korea_standard_concept_coverage_df = (
    korea_fundamentals_standardised_df
    .groupby(["standard_concept","statement_type","core_tier"],dropna=False)
    .agg(
        fact_rows=("fact_key","size"),
        issuer_count=("issuer_id","nunique"),
        security_count=(
            "issuer_id",
            lambda series: 0,
        ),
        filing_count=("rcept_no","nunique"),
        earliest_period=("period_end","min"),
        latest_period=("period_end","max"),
        numeric_fact_share=("reported_value",lambda s:s.notna().mean()),
        period_match_share=("period_type_match","mean"),
        unit_match_share=("unit_family_match","mean"),
    )
    .reset_index()
    if not korea_fundamentals_standardised_df.empty
    else pd.DataFrame()
)

observed = (
    korea_fundamentals_standardised_df
    .groupby("standard_concept",dropna=False)
    .agg(
        issuer_coverage=("issuer_id",lambda s:s.dropna().nunique()),
        security_coverage=(
            "issuer_id",
            lambda series: 0,
        ),
        filing_coverage=("rcept_no",lambda s:s.dropna().nunique()),
        fact_rows=("fact_key","size"),
        first_reporting_date=("period_end","min"),
        last_reporting_date=("period_end","max"),
        first_available_datetime=("available_datetime","min"),
        last_available_datetime=("available_datetime","max"),
        numeric_fact_share=("reported_value",lambda s:s.notna().mean()),
    )
    .reset_index()
    if not korea_fundamentals_standardised_df.empty
    else pd.DataFrame(columns=["standard_concept"])
)

korea_standard_concept_availability_df = global_canonical_schema_df.merge(
    observed,
    on="standard_concept",
    how="left",
)

for col in ["issuer_coverage","security_coverage","filing_coverage","fact_rows"]:
    korea_standard_concept_availability_df[col] = (
        korea_standard_concept_availability_df[col].fillna(0).astype(int)
    )

total_issuers = max(korea_issuer_universe_df["issuer_id"].dropna().nunique(),1)

korea_standard_concept_availability_df["issuer_coverage_rate"] = (
    korea_standard_concept_availability_df["issuer_coverage"] / total_issuers
)

korea_standard_concept_availability_df["coverage_class"] = pd.cut(
    korea_standard_concept_availability_df["issuer_coverage_rate"],
    bins=[-0.001,0.10,0.30,0.60,0.80,1.00],
    labels=["VERY_SPARSE","SPARSE","MODERATE","HIGH","VERY_HIGH"],
)

korea_standard_concept_availability_df["first_reporting_year"] = (
    pd.to_datetime(
        korea_standard_concept_availability_df["first_reporting_date"],
        errors="coerce",
    ).dt.year.astype("Int64")
)

korea_standard_concept_availability_df["last_reporting_year"] = (
    pd.to_datetime(
        korea_standard_concept_availability_df["last_reporting_date"],
        errors="coerce",
    ).dt.year.astype("Int64")
)

korea_mapping_quality_df = pd.DataFrame({
    "metric":[
        "canonical_standard_concepts","korean_source_mapping_rows",
        "mapped_candidate_fact_rows","selected_standardised_fact_rows",
        "unique_standard_concepts_observed","period_type_match_share",
        "unit_family_match_share","unmapped_accounts_in_inventory",
        "automotive_extension_candidates",
    ],
    "value":[
        global_canonical_schema_df["standard_concept"].nunique(),
        len(korea_standard_concept_dictionary_df),
        len(korea_fundamentals_mapped_df),
        len(korea_fundamentals_standardised_df),
        (
            korea_fundamentals_standardised_df["standard_concept"].nunique()
            if not korea_fundamentals_standardised_df.empty else 0
        ),
        (
            korea_fundamentals_standardised_df["period_type_match"].mean()
            if not korea_fundamentals_standardised_df.empty else np.nan
        ),
        (
            korea_fundamentals_standardised_df["unit_family_match"].mean()
            if not korea_fundamentals_standardised_df.empty else np.nan
        ),
        len(korea_unmapped_account_inventory_df),
        len(korea_automotive_extension_candidates_df),
    ],
})

display(korea_filing_coverage_report_df)
display(korea_mapping_quality_df)
display(korea_standard_concept_availability_df.head(100))


# ------------------------------------------------
# ISSUER-CENTRIC LINKAGE QA
# ------------------------------------------------

korea_corp_link_quality_df = (
    pd.DataFrame({
        "metric": [
            "economic_issuer_rows",
            "economic_issuer_count",
            "issuer_rows_with_stock_code",
            "confirmed_corp_issuer_bridges",
            "conflicted_corp_rows",
            "filing_rows",
            "filing_rows_with_issuer_id",
            "filing_rows_missing_issuer_id",
            "distinct_filing_ids",
            "matched_filing_issuer_count",
            "preferred_accounting_source_rows",
        ],
        "value": [
            len(
                korea_economic_issuer_universe_df
            ),
            korea_economic_issuer_universe_df[
                "issuer_id"
            ].nunique(),
            int(
                korea_economic_issuer_universe_df[
                    "stock_code"
                ].notna().sum()
            ),
            len(
                korea_corp_issuer_bridge_df
            ),
            len(
                korea_corp_issuer_conflicts_df
            ),
            len(
                korea_filing_metadata_df
            ),
            int(
                korea_filing_metadata_df[
                    "issuer_id"
                ].notna().sum()
            ),
            int(
                korea_filing_metadata_df[
                    "issuer_id"
                ].isna().sum()
            ),
            (
                korea_filing_metadata_df[
                    "filing_id"
                ].nunique()
                if not korea_filing_metadata_df.empty
                else 0
            ),
            korea_filing_metadata_df[
                "issuer_id"
            ].nunique(),
            len(
                korea_preferred_accounting_source_df
            ),
        ],
    })
)

display(
    korea_corp_link_quality_df
)


,metric,value
0,korean_securities,30
1,korean_issuers,28
2,resolved_dart_corporations,11
3,filing_metadata_rows,322
4,raw_financial_account_rows,92634
5,point_in_time_financial_fact_rows,92634
6,standardised_fact_rows,29678


,metric,value
0,canonical_standard_concepts,100.000000
1,korean_source_mapping_rows,70.000000
2,mapped_candidate_fact_rows,80883.000000
3,selected_standardised_fact_rows,29678.000000
4,unique_standard_concepts_observed,68.000000
5,period_type_match_share,0.972539
6,unit_family_match_share,0.969203
7,unmapped_accounts_in_inventory,1893.000000
8,automotive_extension_candidates,72.000000


,standard_concept,statement_type,expected_period_type,expected_unit_family,core_tier,is_core,aggregation_policy,issuer_coverage,security_coverage,filing_coverage,fact_rows,first_reporting_date,last_reporting_date,first_available_datetime,last_available_datetime,numeric_fact_share,issuer_coverage_rate,coverage_class,first_reporting_year,last_reporting_year
0,revenue,INCOME_STATEMENT,DURATION,MONETARY,1,True,PERIOD_VALUE,11,0,294,602,2019-03-31,2026-03-31,2020-03-29 15:00:00+00:00,2026-03-17 15:00:00+00:00,1.0,0.392857,MODERATE,2019,2026
1,cost_of_revenue,INCOME_STATEMENT,DURATION,MONETARY,1,True,PERIOD_VALUE,11,0,294,602,2019-03-31,2026-03-31,2020-03-29 15:00:00+00:00,2026-03-17 15:00:00+00:00,1.0,0.392857,MODERATE,2019,2026
2,gross_profit,INCOME_STATEMENT,DURATION,MONETARY,1,True,PERIOD_VALUE,11,0,294,602,2019-03-31,2026-03-31,2020-03-29 15:00:00+00:00,2026-03-17 15:00:00+00:00,1.0,0.392857,MODERATE,2019,2026
3,operating_income,INCOME_STATEMENT,DURATION,MONETARY,1,True,PERIOD_VALUE,11,0,294,602,2019-03-31,2026-03-31,2020-03-29 15:00:00+00:00,2026-03-17 15:00:00+00:00,1.0,0.392857,MODERATE,2019,2026
4,profit_before_tax,INCOME_STATEMENT,DURATION,MONETARY,1,True,PERIOD_VALUE,11,0,294,602,2019-03-31,2026-03-31,2020-03-29 15:00:00+00:00,2026-03-17 15:00:00+00:00,1.0,0.392857,MODERATE,2019,2026
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,vehicle_production_volume,OPERATING_METRIC,DURATION,COUNT,3,False,PERIOD_VALUE,0,0,0,0,NaT,NaT,NaT,NaT,NaN,0.000000,VERY_SPARSE,<NA>,<NA>
96,automotive_revenue,SEGMENT,DURATION,MONETARY,3,False,PERIOD_VALUE,0,0,0,0,NaT,NaT,NaT,NaT,NaN,0.000000,VERY_SPARSE,<NA>,<NA>
97,financial_services_revenue,SEGMENT,DURATION,MONETARY,3,False,PERIOD_VALUE,0,0,0,0,NaT,NaT,NaT,NaT,NaN,0.000000,VERY_SPARSE,<NA>,<NA>
98,automotive_debt,SEGMENT,INSTANT,MONETARY,3,False,LATEST_INSTANT,0,0,0,0,NaT,NaT,NaT,NaT,NaN,0.000000,VERY_SPARSE,<NA>,<NA>


,metric,value
0,economic_issuer_rows,29
1,economic_issuer_count,28
2,issuer_rows_with_stock_code,12
3,confirmed_corp_issuer_bridges,11
4,conflicted_corp_rows,0
5,filing_rows,322
6,filing_rows_with_issuer_id,322
7,filing_rows_missing_issuer_id,0
8,distinct_filing_ids,322
9,matched_filing_issuer_count,11


In [ ]:
# 14. OUTPUT CONTRACT
# ------------------------------------------------

block_6_data = {
    # Issuer-centric architecture
    "korea_security_universe_df": korea_security_universe_df,
    "korea_issuer_universe_df": korea_issuer_universe_df,
    "korea_economic_issuer_universe_df": korea_economic_issuer_universe_df,
    "korea_issuer_security_universe_df": korea_issuer_security_universe_df,
    "dart_corporation_registry_df": dart_corporation_registry_df,
    "dart_listed_corporations_df": dart_listed_corporations_df,
    "korea_corporation_bridge_df": korea_corporation_bridge_df,
    "korea_resolved_corporations_df": korea_resolved_corporations_df,
    "korea_unresolved_corporations_df": korea_unresolved_corporations_df,
    "korea_corp_issuer_bridge_candidates_df": korea_corp_issuer_bridge_candidates_df,
    "korea_corp_issuer_bridge_df": korea_corp_issuer_bridge_df,
    "korea_corp_issuer_conflicts_df": korea_corp_issuer_conflicts_df,
    "korea_corp_security_bridge_df": korea_corp_security_bridge_df,
    "korea_preferred_accounting_source_df": korea_preferred_accounting_source_df,
    "korea_entity_relationship_graph_df": korea_entity_relationship_graph_df,

    # Filing discovery and metadata
    "dart_filings_raw_df": dart_filings_raw_df,
    "dart_filing_download_log_df": dart_filing_download_log_df,
    "korea_filing_metadata_df": korea_filing_metadata_df,
    "korea_matched_filings_df": korea_matched_filings_df,
    "korea_unmatched_filings_df": korea_unmatched_filings_df,

    # Financial requests and facts
    "dart_financial_request_grid_df": dart_financial_request_grid_df,
    "dart_financial_accounts_raw_df": dart_financial_accounts_raw_df,
    "dart_financial_download_log_df": dart_financial_download_log_df,
    "dart_financial_receipt_bridge_df": dart_financial_receipt_bridge_df,
    "korea_financial_facts_pit_df": korea_financial_facts_pit_df,

    # Concept mapping
    "korea_source_account_mapping_df": korea_source_account_mapping_df,
    "korea_standard_concept_dictionary_df": korea_standard_concept_dictionary_df,
    "korea_observed_account_mapping_df": korea_observed_account_mapping_df,
    "korea_fundamentals_mapped_df": korea_fundamentals_mapped_df,
    "korea_fundamentals_standardised_df": korea_fundamentals_standardised_df,
    "korea_fundamentals_security_linked_df": korea_fundamentals_security_linked_df,

    # Mapping and coverage QA
    "korea_unmapped_account_inventory_df": korea_unmapped_account_inventory_df,
    "korea_automotive_extension_candidates_df": korea_automotive_extension_candidates_df,
    "korea_filing_coverage_report_df": korea_filing_coverage_report_df,
    "korea_standard_concept_coverage_df": korea_standard_concept_coverage_df,
    "korea_standard_concept_availability_df": korea_standard_concept_availability_df,
    "korea_mapping_quality_df": korea_mapping_quality_df,
    "korea_corp_link_quality_df": korea_corp_link_quality_df,
    "korea_issuer_security_link_quality_df": korea_issuer_security_link_quality_df,
}

print(
    "Block 6 transformations complete."
)

for name in [
    "korea_economic_issuer_universe_df",
    "korea_issuer_security_universe_df",
    "korea_corp_issuer_bridge_df",
    "korea_filing_metadata_df",
    "korea_financial_facts_pit_df",
    "korea_fundamentals_standardised_df",
    "korea_fundamentals_security_linked_df",
]:
    print(
        f"  {name}: "
        f"{len(block_6_data[name]):,} rows"
    )

Block 6 transformations complete.
  korea_economic_issuer_universe_df: 29 rows
  korea_issuer_security_universe_df: 30 rows
  korea_corp_issuer_bridge_df: 11 rows
  korea_filing_metadata_df: 322 rows
  korea_financial_facts_pit_df: 92,634 rows
  korea_fundamentals_standardised_df: 29,678 rows
  korea_fundamentals_security_linked_df: 29,678 rows


In [ ]:
# 15. PERSIST OUTPUTS

def parquet_safe(df):
    out = df.copy()
    for col in out.columns:
        if out[col].dtype == "object":
            non_missing = out[col].dropna()
            if not non_missing.empty and non_missing.map(type).nunique() > 1:
                out[col] = out[col].astype("string")
    return out

def persist(name, df):
    path = BLOCK_6_OUTPUT_DIR / f"{name}.parquet"
    if path.exists() and not OVERWRITE_PERSISTED_OUTPUTS:
        raise FileExistsError(path)
    safe = parquet_safe(df)
    safe.to_parquet(path,index=False,engine="pyarrow",compression="snappy")
    return {
        "table_name":name,
        "path":str(path),
        "row_count":int(len(safe)),
        "column_count":int(len(safe.columns)),
        "columns":list(map(str,safe.columns)),
        "file_size_bytes":int(path.stat().st_size),
        "created_at_utc":datetime.now(timezone.utc).isoformat(),
    }

def load_block_6_outputs():
    with BLOCK_6_MANIFEST_PATH.open("r",encoding="utf-8") as f:
        manifest = json.load(f)
    return {
        item["table_name"]:pd.read_parquet(item["path"])
        for item in manifest["tables"]
    }

if PERSIST_BLOCK_6_OUTPUTS:
    manifest_rows = [persist(name,df) for name,df in block_6_data.items()]

    block_6_manifest = {
        "block":6,
        "block_name":"Korea OpenDART issuer-centric point-in-time fundamentals",
        "created_at_utc":datetime.now(timezone.utc).isoformat(),
        "project_root":str(PROJECT_ROOT),
        "input_manifests":[str(BLOCK_2_MANIFEST_PATH),str(BLOCK_4_MANIFEST_PATH)],
        "output_directory":str(BLOCK_6_OUTPUT_DIR),
        "source_system":"Financial Supervisory Service OpenDART",
        "discovery_start_date":DISCOVERY_START_DATE,
        "discovery_end_date":DISCOVERY_END_DATE,
        "report_codes":REPORT_CODES,
        "canonical_concept_count":int(
            global_canonical_schema_df["standard_concept"].nunique()
        ),
        "tables":manifest_rows,
    }

    with BLOCK_6_MANIFEST_PATH.open("w",encoding="utf-8") as f:
        json.dump(block_6_manifest,f,indent=2,ensure_ascii=False)

    block_6_persistence_report_df = pd.DataFrame(manifest_rows)
    print("Block 6 outputs persisted successfully.")
    print("Manifest:",BLOCK_6_MANIFEST_PATH)
    display(
        block_6_persistence_report_df[
            ["table_name","row_count","column_count","file_size_bytes","path"]
        ]
    )

Block 6 outputs persisted successfully.
Manifest: /content/drive/MyDrive/Colab Notebooks/00 A1 Auto Factor Strategy/data/interim/block_6/block_6_manifest.json


,table_name,row_count,column_count,file_size_bytes,path
0,korea_security_universe_df,30,6,5924,/content/drive/MyDrive/Colab Notebooks/00 A1 A...
1,korea_issuer_universe_df,29,4,4057,/content/drive/MyDrive/Colab Notebooks/00 A1 A...
2,korea_economic_issuer_universe_df,29,4,4057,/content/drive/MyDrive/Colab Notebooks/00 A1 A...
3,korea_issuer_security_universe_df,30,6,5924,/content/drive/MyDrive/Colab Notebooks/00 A1 A...
4,dart_corporation_registry_df,118519,5,4466455,/content/drive/MyDrive/Colab Notebooks/00 A1 A...
5,dart_listed_corporations_df,3978,5,182789,/content/drive/MyDrive/Colab Notebooks/00 A1 A...
6,korea_corporation_bridge_df,30,10,8794,/content/drive/MyDrive/Colab Notebooks/00 A1 A...
7,korea_resolved_corporations_df,11,10,7621,/content/drive/MyDrive/Colab Notebooks/00 A1 A...
8,korea_unresolved_corporations_df,19,10,6987,/content/drive/MyDrive/Colab Notebooks/00 A1 A...
9,korea_corp_issuer_bridge_candidates_df,11,10,7420,/content/drive/MyDrive/Colab Notebooks/00 A1 A...


In [ ]:
# 16. PERSISTENCE VALIDATION
# ------------------------------------------------

if PERSIST_BLOCK_6_OUTPUTS:
    reloaded = load_block_6_outputs()

    required = {
        "korea_economic_issuer_universe_df",
        "korea_issuer_security_universe_df",
        "korea_corp_issuer_bridge_df",
        "korea_corp_issuer_conflicts_df",
        "korea_corp_security_bridge_df",
        "korea_preferred_accounting_source_df",
        "korea_entity_relationship_graph_df",
        "korea_filing_metadata_df",
        "korea_financial_facts_pit_df",
        "korea_standard_concept_dictionary_df",
        "korea_fundamentals_standardised_df",
        "korea_fundamentals_security_linked_df",
        "korea_standard_concept_availability_df",
        "korea_unmapped_account_inventory_df",
        "korea_corp_link_quality_df",
        "korea_issuer_security_link_quality_df",
    }

    missing = (
        required
        - set(
            reloaded
        )
    )

    if missing:
        raise RuntimeError(
            "Persistence validation missing "
            f"{sorted(missing)}"
        )

    validation_rows = []

    for name in sorted(
        required
    ):
        original_rows = len(
            block_6_data[
                name
            ]
        )

        persisted_rows = len(
            reloaded[
                name
            ]
        )

        if (
            original_rows
            != persisted_rows
        ):
            raise RuntimeError(
                f"Row-count mismatch for "
                f"{name}: "
                f"{original_rows} original versus "
                f"{persisted_rows} persisted."
            )

        validation_rows.append({
            "table_name": name,
            "original_rows": original_rows,
            "persisted_rows": persisted_rows,
            "status": "PASSED",
        })

    persisted_facts_df = (
        reloaded[
            "korea_fundamentals_standardised_df"
        ]
    )

    if (
        len(
            persisted_facts_df
        ) > 0
        and persisted_facts_df[
            "issuer_id"
        ].notna().sum()
        == 0
    ):
        raise RuntimeError(
            "Persisted Korean facts contain "
            "no issuer_id values."
        )

    persisted_filings_df = (
        reloaded[
            "korea_filing_metadata_df"
        ]
    )

    if (
        len(
            persisted_filings_df
        ) > 0
        and persisted_filings_df[
            "issuer_id"
        ].notna().sum()
        == 0
    ):
        raise RuntimeError(
            "Persisted Korean filing metadata "
            "contains no issuer_id values."
        )

    if (
        len(
            persisted_filings_df
        ) > 0
        and persisted_filings_df[
            "filing_id"
        ].notna().sum()
        == 0
    ):
        raise RuntimeError(
            "Persisted Korean filing metadata "
            "contains no filing_id values."
        )

    block_6_validation_report_df = (
        pd.DataFrame(
            validation_rows
        )
    )

    display(
        block_6_validation_report_df
    )

    print(
        "Persistence validation passed. "
        "Later modules can load the issuer-centric "
        "Korean fundamentals layer without rerunning "
        "OpenDART collection."
    )

,table_name,original_rows,persisted_rows,status
0,korea_corp_issuer_bridge_df,11,11,PASSED
1,korea_corp_issuer_conflicts_df,0,0,PASSED
2,korea_corp_link_quality_df,11,11,PASSED
3,korea_corp_security_bridge_df,11,11,PASSED
4,korea_economic_issuer_universe_df,29,29,PASSED
5,korea_entity_relationship_graph_df,0,0,PASSED
6,korea_filing_metadata_df,322,322,PASSED
7,korea_financial_facts_pit_df,92634,92634,PASSED
8,korea_fundamentals_security_linked_df,29678,29678,PASSED
9,korea_fundamentals_standardised_df,29678,29678,PASSED


Persistence validation passed. Later modules can load the issuer-centric Korean fundamentals layer without rerunning OpenDART collection.


## Architectural notes

OpenDART accounting facts are canonical at the economic-issuer level. A confirmed
corporation-code-to-issuer bridge controls filing and financial-fact linkage.

The canonical filing table now preserves `rcept_no`, `receipt_number` and
`filing_id`, preventing Korean filing coverage from disappearing in Block 10.

Listed securities are attached only in a separate audited expansion table.

Block 10 should load:

- `korea_economic_issuer_universe_df`
- `korea_issuer_security_universe_df`
- `korea_fundamentals_standardised_df`
- `korea_filing_metadata_df`
- `korea_standard_concept_dictionary_df`
- `korea_preferred_accounting_source_df`
- `korea_entity_relationship_graph_df`
